# I\. Setup and Album Title Inspection

In [1]:
# Standard library imports
import os
from datetime import datetime

# Third-party imports
import numpy as np
import pandas as pd

os.chdir("/work")
print(os.listdir("./pipeline"))


['1.1.TMDB 2015-2025.csv', '2.1.MUSICBRAINZ_mv_tmdb_soundtrack_album_spine_2015_2025.csv', '2.2.MUSICBRAINZ_mv_tmdb_soundtrack_album_artist_bridge_2015_2025.csv', '2.2.MUSICBRAINZ_mv_tmdb_soundtrack_artist_spine_2015_2025.csv', '2.2.MUSICBRAINZ_mv_tmdb_soundtrack_track_spine_2015_2025.csv', '2.2.MUSICBRAINZ_mv_tmdb_soundtrack_wide_track_2015_2025.csv', '3.2.Albums_official_df.csv', '3.2.Artists_official_df.csv', '3.2.Tracks_official_df.csv', '3.2.Wide_official_df.csv', '3.5.Albums_exploded_genre.csv', '3.5.Artists_exploded_genre.csv', '3.5.Tracks_exploded_genre.csv', '3.5.Wide_exploded_genre.csv', '3.6.Albums_vote_count_analysis.csv', '3.6.Artists_vote_count_analysis.csv', '3.6.Tracks_vote_count_analysis.csv', '3.6.Wide_vote_count_analysis.csv', '3.7.Albums_composer_analysis.csv', '3.7.Artists_composer_analysis.csv', '3.7.Tracks_composer_analysis.csv', '3.7.Wide_composer_analysis.csv', '4.1.1.Albums_awards_appended.csv', '4.1.1.Artists_awards_appended.csv', '4.1.1.Tracks_awards_appende

In [2]:
# Load the albums dataframe

album_df = pd.read_csv("./pipeline/3.4.Albums_canonical_identified_df.csv")
display(album_df.shape)
display(album_df.columns)
display(album_df.head())

(4771, 69)

Index(['tmdb_id', 'film_title', 'film_adult', 'film_runtime_min',
       'film_genres', 'film_rating', 'film_vote_count', 'film_mpaa_rating',
       'film_original_title', 'film_language_name', 'film_imdb_id',
       'film_wikidata_id', 'film_countries', 'film_year', 'film_release_date',
       'film_popularity', 'film_budget', 'film_revenue', 'film_studios',
       'film_director', 'film_soundtrack_composer_raw', 'film_top_cast',
       'film_keywords', 'film_ingested_at', 'match_method', 'soundtrack_type',
       'notes', 'matched_at', 'release_group_id', 'release_group_mbid',
       'album_title', 'rg_primary_type', 'rg_secondary_types', 'release_id',
       'release_mbid', 'release_title', 'release_status', 'release_packaging',
       'barcode', 'release_language', 'release_script', 'release_comment',
       'album_us_release_date', 'album_us_release_year',
       'album_us_release_month_min_observed',
       'album_us_release_day_min_observed', 'us_date_has_missing_month',
       

,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,release_meta_amazon_asin,release_meta_cover_art_presence,release_status_clean,keep_row,is_canonical_soundtrack,is_canonical_songtrack,canonical_rule,canonical_songtrack_rule,days_since_film_release,days_since_album_release
0,216015,Fifty Shades of Grey,False,125,"Drama, Romance, Thriller",5.875,12222,R,Fifty Shades of Grey,English,...,NaN,present,Official,True,False,True,NaN,multi_album_soundtrack_title,4020,4021.0
1,216015,Fifty Shades of Grey,False,125,"Drama, Romance, Thriller",5.875,12222,R,Fifty Shades of Grey,English,...,B00S9303ZM,present,Official,True,True,False,multi_album_score_title,NaN,4020,4014.0
2,150689,Cinderella,False,105,"Romance, Fantasy, Family, Drama",6.826,7294,PG,Cinderella,English,...,B00THRE5TO,present,Official,True,True,False,single_album_film,NaN,3997,3993.0
3,281957,The Revenant,False,157,"Western, Drama, Adventure",7.540,18928,R,The Revenant,English,...,B018THTR7M,present,Official,True,True,False,single_album_film,NaN,3703,3703.0
4,140607,Star Wars: The Force Awakens,False,136,"Adventure, Action, Science Fiction",7.300,20206,PG-13,Star Wars: The Force Awakens,English,...,B014V6JIQK,present,Official,True,True,True,fallback_imdb_match,multi_album_soundtrack_title,3713,4061.0


Let's look closely at the available tags

In [3]:
null_rg_tags = album_df['rg_tags_text'].isna().sum()
print(f"% of rg tags that are null: {null_rg_tags/len(album_df):.3f}")
  # 80.5% of rg tags are null. So the % of albums with tag genres is quite low, but it's the 
  # closest approximation to genre that we have

null_release_tags = album_df['release_tags_text'].isna().sum()
print(f"% of release tags that are null: {null_release_tags/len(album_df):.3f}")
  # 92% of release tags are null. This is an even higher null percentage

null_label_tags = album_df['label_tags_text'].isna().sum()
print(f"% of label tags that are null: {null_label_tags/len(album_df):.3f}")
  # 42% of label tags are null -- so more than half are available. However, I'm not sure how useful
  # label tags would be for our analysis

% of rg tags that are null: 0.805
% of release tags that are null: 0.923
% of label tags that are null: 0.419


Findings: MusicBrainz tag coverage is uneven\. Roughly 80\.5% of release\-group tags and 92\.3% of release\-level tags are missing, limiting their usefulness as genre proxies\. Label tags are more populated \(only 41\.9% missing\), but their analytical value is unclear\. Overall, tag\-based genre enrichment would be sparse and potentially inconsistent\.

In [4]:
# Count the number of albums with rg tags but not release tags
release_na = len(album_df[(album_df['rg_tags_text'].notna()) & (album_df['release_tags_text'].isna())])

# Count the number of albums with release tags but not rg tags
rg_na = len(album_df[(album_df['rg_tags_text'].isna()) & (album_df['release_tags_text'].notna())])

# Count the number of albums with values on both
neither_na = len(album_df[(album_df['rg_tags_text'].notna()) & (album_df['release_tags_text'].notna())])

# Count the number where they are both na
both_na = len(album_df[(album_df['rg_tags_text'].isna()) & (album_df['release_tags_text'].isna())])

print(f"Both null: {both_na}: {both_na/len(album_df):.3f}")
print(f"Release Group null / Release has value: {rg_na}: {rg_na/len(album_df):.3f}")
print(f"Release null / Release Group has value: {release_na}: {release_na/len(album_df):.3f}")
print(f"Neither null: {neither_na}: {neither_na/len(album_df):.3f}")



Both null: 3739: 0.784
Release Group null / Release has value: 103: 0.022
Release null / Release Group has value: 665: 0.139
Neither null: 264: 0.055


Findings: 14% of rg\_tags\_text have values where Release has not, and 5\.5% populate both\. Therefore, it makes sense to treat rg\_tags\_text as the primary source for tags\. However, about 2% of albums contain release tags but no rg\_tags, so combining both may provide some marginal value\. 

In [5]:
album_df['label_tags_text'].value_counts()

label_tags_text
soundtrack | soundtrack label                                                              332
burbank | california | clean up | score | soundtrack label | split me | us | soundtrack    240
indian | filmi | indian pop                                                                197
soundtrack                                                                                 194
label by lytron                                                                            172
                                                                                          ... 
soundtrack label | burbank | california | clean up | score | soundtrack | split me | us      1
record company logo | clean up                                                               1
anime | japan | japanese                                                                     1
_add patreon prerelease | has full bandcamp discography purchase                             1
vgm | video game music | soundtrac

Findings: A long list of random values with occasional genre information\. However, leveraging genres at the label\-level \(a fundamentally different entity from the album\) will likely degrade the quality of our genre signal, and potentially introduce a lot of garbage data if we're not careful\. So, we will not use label tags\.

# II\. Album Genre exploding

As mentioned in an early notebook, we want to simplify genre into seven buckets, and append these as features to the album\_df

Recap: We want to take the free\-form rg\_tags\_text field, split it into individual tags, and map any tags that matches our curated synonym list into a small set of canonical genres \(7 max\)\. Then we roll that back up to the album grain \(release\_group\_mbid\) to produce a compact genre\_append table with boolean flags \(e\.g\., electronic=True, rock=False, etc\.\) that can be merged into both album\_df and wide\_df\.

### II\.1 Genre thesaurus

Export the data so we can eyeball the different tags

In [6]:
album_df[album_df['release_tags_text'].notna()].to_csv('./tmp/release_tags.csv')

Leveraging an LLM to help organize the data, we come up with the following dictionaries that map to 7 distinct album genres \(7 seemed to be the optimal number \-\- proposed categorizations started looking weird at 8\+ buckets\)

In [7]:
CANONICAL_GENRE_MAP = {
    "classical_orchestral": [
        # Core classical buckets
        "classical", "modern classical", "contemporary classical",
        "cinematic classical", "orchestral", "instrumental",
        "baroque", "opera", "chamber music",
        "minimalism", "neoclassicism", "classical crossover",
        # Helpful additions seen in release tags (style, not role)
        "neo-classical", "post-classical",
        "orchestral music", "string quartet"
    ],

    "electronic": [
        # Broad electronic umbrella
        "electronic", "electronica", "electro", "edm",
        "techno", "house", "deep house", "progressive house",
        "trance", "dubstep", "idm", "ebm",
        "electro-industrial", "electroacoustic", "industrial",
        # Additions that show up heavily in soundtrack-style tags
        "synthwave", "retrowave",
        "darkwave", "minimal synth",
        "future bass", "breakcore", "chiptune"
    ],

    "ambient_experimental": [
        # Ambient + experimental umbrella
        "ambient", "dark ambient", "drone", "downtempo",
        "chillout", "lounge", "new age",
        "experimental", "avant-garde", "noise",
        "sound art", "atmospheric", "cinematic",
        # Texture/style additions (common in release tags)
        "darksynth", "soundscape",
        "musique concrete", "sound collage",
        "experimental electronic", "cinematic ambient",
        "atmospheric ambient"
    ],

    "rock": [
        "rock", "alternative rock", "indie rock", "post-rock",
        "hard rock", "classic rock", "punk", "post-punk",
        "grunge", "psychedelic rock", "shoegaze",
        "progressive rock", "krautrock",
        "metal", "heavy metal", "death metal", "black metal",
        "alternative metal", "progressive metal",
        # Minor additions that appear in release tag space
        "instrumental rock", "arena rock"
    ],

    "pop": [
        "pop", "electropop", "synth-pop", "dance-pop",
        "indie pop", "art pop", "chamber pop",
        "europop", "traditional pop", "pop rock"
    ],

    "hip_hop_rnb": [
        "hip hop", "hip-hop", "hiphop", "rap",
        "trap", "grime", "drill", "gangsta rap", "pop rap",
        "r&b", "rhythm and blues", "contemporary r&b",
        "neo soul", "soul", "funk", "disco"
    ],

    "world_folk": [
        "world", "folk", "contemporary folk", "alternative folk",
        "americana", "country", "outlaw country", "bluegrass",
        "latin", "latin pop", "reggae", "ska",
        "afrobeat", "afrobeats", "afroswing",
        "bhangra", "celtic", "polka",
        # Useful add from release tags
        "indian classical"
    ]
}


### II\.2 Reverse lookup dictionary

In [8]:
# ---- 1. Build reverse lookup: raw_tag -> canonical genre ----
# (If a raw tag appears in multiple canons, we keep the first; you can change this later.)
tag_to_canon = {}
for canon, syns in CANONICAL_GENRE_MAP.items():
    for s in syns:
        key = str(s).strip().lower()
        if key and key not in tag_to_canon:
            tag_to_canon[key] = canon

GENRE_COLS = list(CANONICAL_GENRE_MAP.keys())

display(tag_to_canon)
# Reverse lookup created:
# {'classical': 'classical_orchestral',
# 'modern classical': 'classical_orchestral',
# 'contemporary classical': 'classical_orchestral', ..

{'classical': 'classical_orchestral',
 'modern classical': 'classical_orchestral',
 'contemporary classical': 'classical_orchestral',
 'cinematic classical': 'classical_orchestral',
 'orchestral': 'classical_orchestral',
 'instrumental': 'classical_orchestral',
 'baroque': 'classical_orchestral',
 'opera': 'classical_orchestral',
 'chamber music': 'classical_orchestral',
 'minimalism': 'classical_orchestral',
 'neoclassicism': 'classical_orchestral',
 'classical crossover': 'classical_orchestral',
 'neo-classical': 'classical_orchestral',
 'post-classical': 'classical_orchestral',
 'orchestral music': 'classical_orchestral',
 'string quartet': 'classical_orchestral',
 'electronic': 'electronic',
 'electronica': 'electronic',
 'electro': 'electronic',
 'edm': 'electronic',
 'techno': 'electronic',
 'house': 'electronic',
 'deep house': 'electronic',
 'progressive house': 'electronic',
 'trance': 'electronic',
 'dubstep': 'electronic',
 'idm': 'electronic',
 'ebm': 'electronic',
 'electr

To maximize the limited genre signal available in MusicBrainz, we build a simple tag “union” field by combining rg\_tags\_text \(release\-group tags\) and release\_tags\_text \(release\-level tags\)\. This creates a single consolidated tag string per album, preserving whichever tag source is present and merging both when available\.

In [9]:
# ---- 2. Build a base (rg_tags_text and release_tags_text union)
album_genre_base = album_df.loc[:, ["release_group_mbid", "rg_tags_text", "release_tags_text"]].copy()

def _join_tags(a, b):
    a = "" if pd.isna(a) else str(a).strip()
    b = "" if pd.isna(b) else str(b).strip()
    if a and b:
        return f"{a} | {b}"
    return a or b or None

album_genre_base["tags_text_union"] = album_genre_base.apply(
    lambda r: _join_tags(r["rg_tags_text"], r["release_tags_text"]),
    axis=1
)

### II\.3 Build a genre\_append table

Next, we convert the combined tag string into an analyzable “long” format\. We first split tags\_text\_union into a list of individual tags, then use explode\(\) to expand that list so each tag becomes its own row \(i\.e\., one row per album–tag pair\)\. This makes it easy to count, filter, and map tags consistently\.

In [10]:
# ---- 3. Explode the tags -- creates one row per tag
tags_long = album_genre_base.loc[
    album_genre_base["tags_text_union"].notna() & (album_genre_base["tags_text_union"] != ""), :].copy()

tags_long["raw_tag"] = tags_long["tags_text_union"].str.split(" | ", regex=False)
tags_long = tags_long.explode("raw_tag", ignore_index=True)

# Normalize
tags_long["raw_tag"] = tags_long["raw_tag"].str.strip().str.lower()

display(tags_long.head())


,release_group_mbid,rg_tags_text,release_tags_text,tags_text_union,raw_tag
0,85852629-ce4c-4bb8-8d4a-50299d56a06c,contemporary r&b | pop | blues | contemporary ...,soundtrack,contemporary r&b | pop | blues | contemporary ...,contemporary r&b
1,85852629-ce4c-4bb8-8d4a-50299d56a06c,contemporary r&b | pop | blues | contemporary ...,soundtrack,contemporary r&b | pop | blues | contemporary ...,pop
2,85852629-ce4c-4bb8-8d4a-50299d56a06c,contemporary r&b | pop | blues | contemporary ...,soundtrack,contemporary r&b | pop | blues | contemporary ...,blues
3,85852629-ce4c-4bb8-8d4a-50299d56a06c,contemporary r&b | pop | blues | contemporary ...,soundtrack,contemporary r&b | pop | blues | contemporary ...,contemporary r b
4,85852629-ce4c-4bb8-8d4a-50299d56a06c,contemporary r&b | pop | blues | contemporary ...,soundtrack,contemporary r&b | pop | blues | contemporary ...,dance-pop


At this stage we translate messy, user\-generated tag strings into a small, consistent genre vocabulary\. We map each raw\_tag to a canonical\_genre using our lookup dictionary, then drop tags that don’t map cleanly \(i\.e\., noise or overly specific “junk” tags\)\. The resulting tags\_mapped table is our curated set of album–genre assignments\.

In [11]:
# ---- 4. Map raw tags to canonical genre
tags_long['canonical_genre'] = tags_long['raw_tag'].map(tag_to_canon)  # This will result in lots of empty mappings for junk tags
tags_mapped = tags_long.dropna(subset = ['canonical_genre']).copy()

# Confirm mapping
display(tags_mapped.head())

,release_group_mbid,rg_tags_text,release_tags_text,tags_text_union,raw_tag,canonical_genre
0,85852629-ce4c-4bb8-8d4a-50299d56a06c,contemporary r&b | pop | blues | contemporary ...,soundtrack,contemporary r&b | pop | blues | contemporary ...,contemporary r&b,hip_hop_rnb
1,85852629-ce4c-4bb8-8d4a-50299d56a06c,contemporary r&b | pop | blues | contemporary ...,soundtrack,contemporary r&b | pop | blues | contemporary ...,pop,pop
4,85852629-ce4c-4bb8-8d4a-50299d56a06c,contemporary r&b | pop | blues | contemporary ...,soundtrack,contemporary r&b | pop | blues | contemporary ...,dance-pop,pop
5,85852629-ce4c-4bb8-8d4a-50299d56a06c,contemporary r&b | pop | blues | contemporary ...,soundtrack,contemporary r&b | pop | blues | contemporary ...,electronic,electronic
6,85852629-ce4c-4bb8-8d4a-50299d56a06c,contemporary r&b | pop | blues | contemporary ...,soundtrack,contemporary r&b | pop | blues | contemporary ...,hip hop,hip_hop_rnb


With tags normalized and mapped to a controlled set of canonical genres, we now materialize the result back at the album level\. This cell pivots the long tag table into a wide “genre append” table—one row per release\_group\_mbid, with a boolean flag for each canonical genre indicating whether that genre appears anywhere in the album’s tags\.

In [12]:
# ---- 5. Build the genre_append table at album grain (boolean flag per canonical genre)
genre_append = tags_mapped.assign(present = 1).pivot_table(
    index = 'release_group_mbid',
    columns = 'canonical_genre',
    values = 'present',
    aggfunc = 'max',
    fill_value = 0
).reset_index()

# Convert to boolean flags
for g in GENRE_COLS:
    genre_append[g] = genre_append[g].astype(bool)

display(genre_append)

canonical_genre,release_group_mbid,ambient_experimental,classical_orchestral,electronic,hip_hop_rnb,pop,rock,world_folk
0,00a44fb9-8338-46b1-9b19-1718fba212a8,False,True,False,False,False,False,False
1,00b0a5f5-7a6b-430f-a733-92a11ba04ddd,False,False,True,False,True,False,False
2,0138854e-c21a-4102-88a9-306c142aa66b,False,False,False,False,False,False,True
3,019a4e32-97d7-44b4-93a9-cc3b9f88b6ef,False,False,False,False,True,False,False
4,01c0ab6d-cae3-4c93-abc8-764194206772,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...
742,ff236112-b2c9-4e72-a247-e05b7781f029,False,True,True,True,True,False,True
743,ff624422-97fa-4a8c-bee7-f23a4f0206be,False,False,False,False,False,True,False
744,ff913409-438d-4714-a16e-787c0fc9431c,True,True,True,False,False,False,False
745,ffda7540-c5a5-44e7-801f-fcbfda2a2dcc,False,False,False,False,False,True,True


### II\.4 Append the genre table

### Album

Finally, we attach the genre flags back onto the main album table\. We use a left join on release\_group\_mbid so every album stays in the dataset—even those with no usable tags—while tagged albums gain boolean genre columns for downstream analysis and modeling\.

In [13]:
# Merge genre flags onto album_df (album grain: release_group_mbid)
# Left join so we don't drop any albums that don't have genre tags.
album_merged_df = album_df.merge(
    genre_append,
    on="release_group_mbid",
    how="left"
)

print("album_df rows:", len(album_df),"album_merged_df rows:", len(album_merged_df))
print("Unique release_group_mbids in album_merged_df:", album_merged_df["release_group_mbid"].nunique())

album_df rows: 4771 album_merged_df rows: 4771
Unique release_group_mbids in album_merged_df: 4771


In [14]:
# 3) Quick inspection
print("\nGenre columns added:", len(GENRE_COLS))
print("Albums with ANY genre flagged:", (album_merged_df[GENRE_COLS].any(axis=1)).sum())
print("Albums with NO genre flagged:", (~album_merged_df[GENRE_COLS].any(axis=1)).sum())
print("% albums with ANY genre flagged:", (album_merged_df[GENRE_COLS].any(axis=1)).sum()/len(album_merged_df))

# How many genres per album (helps detect overly-broad tagging)
genre_ct = album_merged_df[GENRE_COLS].sum(axis=1)
print("\nGenres per album (distribution):")
display(genre_ct.value_counts().sort_index().head(25))

# Top genres overall (coverage)
top_genres = album_merged_df[GENRE_COLS].sum().sort_values(ascending=False)
print("\nTop genres (albums flagged):")
display(top_genres.head(25))


Genre columns added: 7
Albums with ANY genre flagged: 747
Albums with NO genre flagged: 4024
% albums with ANY genre flagged: 0.15657094948648081

Genres per album (distribution):


0    4024
1     444
2     185
3      87
4      26
5       4
6       1
Name: count, dtype: int64


Top genres (albums flagged):


classical_orchestral    294
electronic              249
pop                     208
ambient_experimental    155
rock                    153
hip_hop_rnb              95
world_folk               51
dtype: object

Findings: Genre coverage remains sparse: only 747 albums \(~15\.7%\) have at least one mapped canonical genre, while the majority \(4,024\) have none\. Most tagged albums carry just one or two genres\. The most common mapped genres are classical\_orchestral, electronic, and pop, suggesting the tag signal skews toward traditional score and mainstream styles rather than niche categories\.

In [15]:
# Spot check a few rows where genres exist
display(
    album_merged_df.loc[album_merged_df[GENRE_COLS].any(axis=1),
                 ["release_group_mbid", "album_title"] + GENRE_COLS]
    .head(20)
)

,release_group_mbid,album_title,classical_orchestral,electronic,ambient_experimental,rock,pop,hip_hop_rnb,world_folk
0,85852629-ce4c-4bb8-8d4a-50299d56a06c,Fifty Shades of Grey: Original Motion Picture ...,False,True,False,False,True,True,False
2,412fdd65-c3f8-40fe-a420-8b438e048915,Cinderella,True,False,False,False,False,False,False
3,5d5a40d3-8aab-41a8-8cfb-b97688a5c10c,The Revenant: Original Motion Picture Soundtrack,True,False,True,False,False,False,False
4,405fd3c5-0a45-456a-b853-6f734d3b57aa,Star Wars: The Force Awakens: Original Motion ...,True,False,False,False,False,False,False
6,771042e4-5d18-4f47-a2bf-51bc51b37c2c,Sicario: Original Motion Picture Soundtrack,True,True,False,False,False,False,False
11,57bf99c5-b4ff-439f-82fd-f4ebbffb1846,The Hateful Eight,True,False,False,True,False,False,True
12,e0266ee3-7aed-47a2-8433-b6d506870935,Ex_Machina: Original Motion Picture Soundtrack,False,True,False,False,False,False,False
13,87789c04-aeec-48f8-b416-b96b031b2ab9,Ant‐Man: Original Motion Picture Soundtrack,True,False,False,False,False,False,False
14,528bd202-fa2d-4abc-ba3b-d5dc3ce156f5,It Follows: Original Motion Picture Soundtrack,False,True,True,False,False,False,False
17,01f9eac9-2add-4668-94fc-5580623980b2,Creed: Original Motion Picture Score,True,False,False,False,False,False,False


### Wide Table

In [16]:
# Load the albums dataframe

wide_df = pd.read_csv("./pipeline/3.4.Wide_canonical_identified_df.csv")
display(wide_df.shape)
display(wide_df.columns)
display(wide_df.head())

/tmp/ipykernel_445/816116344.py:3: DtypeWarning: Columns (58) have mixed types. Specify dtype option on import or set low_memory=False.
  wide_df = pd.read_csv("./pipeline/3.4.Wide_canonical_identified_df.csv")


(78992, 94)

Index(['tmdb_id', 'film_title', 'film_adult', 'film_runtime_min',
       'film_genres', 'film_rating', 'film_vote_count', 'film_mpaa_rating',
       'film_original_title', 'film_language_name', 'film_imdb_id',
       'film_wikidata_id', 'film_countries', 'film_year', 'film_release_date',
       'film_popularity', 'film_budget', 'film_revenue', 'film_studios',
       'film_director', 'film_soundtrack_composer_raw', 'film_top_cast',
       'film_keywords', 'film_ingested_at', 'release_group_id',
       'release_group_mbid', 'release_id', 'release_mbid', 'album_title',
       'rg_primary_type', 'rg_secondary_types', 'match_method',
       'album_soundtrack_type', 'album_notes', 'album_matched_at',
       'album_us_release_date', 'album_us_release_year',
       'album_us_release_month_min_observed',
       'album_us_release_day_min_observed', 'us_date_has_missing_month',
       'us_date_has_missing_day', 'us_any_event_missing_month',
       'us_any_event_missing_day', 'release_title', 'rel

,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,work_ids_text,work_titles_text,composer_names_text,lyricist_names_text,is_canonical_soundtrack,is_canonical_songtrack,canonical_rule,canonical_songtrack_rule,days_since_film_release,days_since_album_release
0,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,NaN,NaN,NaN,NaN,True,False,single_album_film,NaN,1225,NaN
1,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,NaN,NaN,NaN,NaN,True,False,single_album_film,NaN,1225,NaN
2,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,NaN,NaN,NaN,NaN,True,False,single_album_film,NaN,2918,NaN
3,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,NaN,NaN,NaN,NaN,True,False,single_album_film,NaN,2918,NaN
4,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,NaN,NaN,NaN,NaN,True,False,single_album_film,NaN,2918,NaN


In [17]:
GENRE_COLS = list(CANONICAL_GENRE_MAP.keys())

# The column names in genre_append worked fine when merging with the album table
# However, when we merge with the wide table, we really should append album to it
genre_append_renamed = genre_append.rename(
    columns={g: f"album_{g}" for g in GENRE_COLS}
)

wide_before = wide_df.shape

wide_mrg_df = wide_df.merge(
    genre_append_renamed,   # release_group_mbid + album_genre__*
    on="release_group_mbid",
    how="left",
    validate="m:1"
)

print("wide_df shape before:", wide_before)
print("wide_df shape after: ", wide_mrg_df.shape)
display(wide_mrg_df.columns)


wide_df shape before: (78992, 94)
wide_df shape after:  (78992, 101)


Index(['tmdb_id', 'film_title', 'film_adult', 'film_runtime_min',
       'film_genres', 'film_rating', 'film_vote_count', 'film_mpaa_rating',
       'film_original_title', 'film_language_name',
       ...
       'canonical_songtrack_rule', 'days_since_film_release',
       'days_since_album_release', 'album_ambient_experimental',
       'album_classical_orchestral', 'album_electronic', 'album_hip_hop_rnb',
       'album_pop', 'album_rock', 'album_world_folk'],
      dtype='object', length=101)

In [18]:
# Remember that we renamed the columns for wide
WIDE_GENRE_COLS = [f"album_{g}" for g in GENRE_COLS]

# Coverage check
has_any_genre = wide_mrg_df[WIDE_GENRE_COLS].any(axis=1)
print("Wide rows with >=1 genre:", has_any_genre.sum())
print("Pct wide rows with >=1 genre:", has_any_genre.mean())

album_level = wide_mrg_df.groupby("release_group_mbid")[WIDE_GENRE_COLS].max()
print("Release groups with >=1 genre:", album_level.any(axis=1).sum())
print("Pct release groups with >=1 genre:", album_level.any(axis=1).mean())

Wide rows with >=1 genre: 13403
Pct wide rows with >=1 genre: 0.1696754101681183
Release groups with >=1 genre: 743
Pct release groups with >=1 genre: 0.15609243697478992


In [19]:
canon_rows = wide_mrg_df["is_canonical_soundtrack"] == True
song_rows = wide_mrg_df["is_canonical_songtrack"] == True

print("Pct canonical rows with >=1 genre:",
      wide_mrg_df.loc[canon_rows, WIDE_GENRE_COLS].any(axis=1).mean())

print("Pct non-canonical rows with >=1 genre:",
      wide_mrg_df.loc[~canon_rows, WIDE_GENRE_COLS].any(axis=1).mean())

print("Pct song-canonical rows with >=1 genre:",
      wide_mrg_df.loc[song_rows, WIDE_GENRE_COLS].any(axis=1).mean())



Pct canonical rows with >=1 genre: 0.16608002589311
Pct non-canonical rows with >=1 genre: 0.2247469531088618
Pct song-canonical rows with >=1 genre: 0.28025974025974026


In [20]:
# Spot check a few rows where genres exist
display(
    wide_mrg_df.loc[wide_mrg_df[WIDE_GENRE_COLS].any(axis=1),
                 ["release_group_mbid", "album_title", "track_id", "track_title"] + WIDE_GENRE_COLS]
    .sample(20)
)

,release_group_mbid,album_title,track_id,track_title,album_classical_orchestral,album_electronic,album_ambient_experimental,album_rock,album_pop,album_hip_hop_rnb,album_world_folk
21917,4e2531c6-352e-414f-b2f4-4d419fd5c718,We the Animals,26653609,Look at Me,False,True,True,False,True,False,True
24544,e117e2c3-8132-4f7b-b475-1a7cae9b1c63,Knife + Heart,27662525,Corridor,False,True,False,True,True,True,False
13664,a7c19577-7fb9-479f-a4b9-7456399daf5f,Bitter Lake: The Original Soundtrack,23840212,Fickles of Time (instrumental),True,False,True,False,False,False,False
23839,bee0a7ed-a575-4dcb-9f7b-0c4530c1019c,The LEGO® Movie 2: Original Motion Picture Score,27410064,Introducting Queen Watevra Wa’Nabi,True,False,False,False,False,False,False
62898,84aa00b9-c920-4d5b-9dff-33c737a37299,Confidenza: Music for the Film by Daniele Luch...,46638202,Four Ways in Time,True,True,True,False,False,False,False
24689,27d9e9e1-0422-4fc9-9c47-b4f0aa180456,McQueen,27747828,Search for the Golden Fleece: A Satire Against...,True,False,False,False,False,False,False
74036,e9dfea93-6076-4e8e-b45e-1fb19c0c7ab3,TRON: Ares: Original Motion Picture Soundtrack,52862552,Shadow Over Me,False,True,True,False,False,False,False
5688,bd88f17e-8d74-4436-a4a9-989b69b45f3c,The BFG: Original Motion Picture Soundtrack,20911359,Overture,True,False,False,False,False,False,False
47896,7c5a1e22-ca00-4576-81f3-1f8abb5638ba,The Informer (Original Motion Picture Soundtrack),38777882,The Burning Game,True,False,False,False,False,False,False
6505,f03fe9b7-08d2-4812-8b41-1a157f104e88,Kubo and the Two Strings (Original Motion Pict...,21142460,United-Divided,True,False,False,False,False,False,False


# III\. Other tag inspection

In [21]:
# Load the albums dataframe

artists_df = pd.read_csv("./pipeline/3.4.Artists_canonical_identified_df.csv")
tracks_df = pd.read_csv("./pipeline/3.4.Tracks_canonical_identified_df.csv")

display(artists_df.columns)
display(tracks_df.columns)

Index(['artist_id', 'artist_mbid', 'name', 'sort_name', 'artist_type',
       'gender', 'artist_comment', 'begin_date_year', 'begin_date_month',
       'begin_date_day', 'end_date_year', 'end_date_month', 'end_date_day',
       'area_name', 'area_mbid', 'begin_area_name', 'begin_area_mbid',
       'end_area_name', 'end_area_mbid', 'aliases_text', 'artist_tags_text',
       'isni_text', 'ipi_text', 'film_ct_in_spine',
       'soundtrack_release_group_ct_in_spine'],
      dtype='object')

Index(['tmdb_id', 'film_title', 'release_group_id', 'release_id',
       'match_method', 'album_us_release_date', 'us_date_has_missing_month',
       'us_date_has_missing_day', 'medium_id', 'disc_number', 'medium_format',
       'track_id', 'track_number', 'track_title', 'track_length_ms',
       'recording_id', 'recording_mbid', 'recording_title',
       'recording_length_ms', 'recording_artist_credit',
       'recording_first_release_year', 'recording_first_release_month',
       'recording_first_release_day', 'isrcs_text', 'recording_tags_text',
       'work_ids_text', 'work_titles_text', 'composer_names_text',
       'lyricist_names_text'],
      dtype='object')

In [22]:
null_artist_tags = artists_df['artist_tags_text'].isna().sum()
print(f"% of artist tags that are null: {null_artist_tags/len(artists_df):.3f}")
# 60% of tags are null

artists_df['artist_tags_text'].to_csv('./tmp/artist_tags.csv')

% of artist tags that are null: 0.602


Findings: Artists tags might be an interesting attribute to explode\. However, they are far noisier than album or release tags, often conflating genre with role, geography, and miscellaneous metadata, which limits their usefulness as a clean genre signal without heavy preprocessing\.

In [23]:
null_rec_tags = tracks_df['recording_tags_text'].isna().sum()
print(f"% of recording tags that are null: {null_rec_tags/len(tracks_df):.3f}")
  # 96% of recording tags are null. This column is unusable


% of recording tags that are null: 0.962


Findings: Track\-level tags are pretty useless\. They are mostly null\.

# IV\. Exploding Artists tags

### IV\.1 Create a mapping dictionary for artist roles

We took all the possible artist tag values, sent it to an LLM to process\. There were a number of categorization options, so we decided on using a "type of artist" categorization scheme

In [24]:
CANONICAL_ARTIST_ROLE_MAP = {
    "media_composer_orchestral": [
        # Core media composition roles
        "film composer", "television composer", "tv composer",
        "video game composer", "game composer", "vgm",
        "score", "soundtrack", "film score",

        # Classical / orchestral context
        "composer", "orchestral", "orchestra",
        "modern classical", "contemporary classical",
        "cinematic classical", "classical composer"
    ],

    "pop_vocalist": [
        "pop", "pop rock", "dance-pop", "electropop",
        "singer", "vocalist", "female vocals", "male vocalist",
        "adult contemporary", "teen pop", "pop soul"
    ],

    "rock_artist_band": [
        "rock", "alternative rock", "indie rock",
        "classic rock", "hard rock",
        "punk", "post-punk", "grunge",
        "psychedelic rock", "shoegaze",
        "band", "guitarist"
    ],

    "electronic_producer": [
        "electronic", "electronica", "edm",
        "techno", "house", "deep house", "progressive house",
        "trance", "dubstep", "idm",
        "synth", "synthwave", "retrowave",
        "producer", "dj"
    ],

    "jazz_traditional": [
        "jazz", "blues", "soul jazz", "hard bop",
        "bebop", "post-bop", "big band",
        "swing", "vocal jazz", "instrumental jazz"
    ],

    "world_folk_regional": [
        "world", "folk", "traditional",
        "americana", "country", "bluegrass",
        "latin", "latin pop",
        "afrobeat", "afrobeats",
        "bhangra", "celtic"
    ]
}


### IV\.2 Append and merge with artists\_df

We'll translate and merge this in a similar way to the album tags\.

In [25]:
# ---- 1. Build reverse lookup: raw_tag -> canonical artist role cluster ----
# (If a raw tag appears in multiple canons, we keep the first; you can change this later.)
artist_tag_to_canon = {}
for canon, syns in CANONICAL_ARTIST_ROLE_MAP.items():
    for s in syns:
        key = str(s).strip().lower()
        if key and key not in artist_tag_to_canon:
            artist_tag_to_canon[key] = canon

ARTIST_ROLE_COLS = list(CANONICAL_ARTIST_ROLE_MAP.keys())

# ---- 2. Build a base (artist_tags_text and artist_tag_text union, if you have both) ----
# Assumes artist_df has at least: artist_id, artist_tags_text
# If you only have one tags column, this still works (it just passes through).
artist_role_base = artists_df.loc[:, ["artist_id", "artist_tags_text"]].copy()

Next we repeat the same “tag explosion” procedure used for album genres, but at the artist level\. We split each artist\_tags\_text string into individual tags, then explode\(\) the list so we get one row per artist–tag pair, followed by light normalization for consistent mapping\.

In [26]:
# ---- 3. Explode the tags -- creates one row per tag ----
tags_long_artist = artist_role_base.loc[
    artist_role_base["artist_tags_text"].notna() & (artist_role_base["artist_tags_text"] != ""),
    :
].copy()

tags_long_artist["raw_tag"] = tags_long_artist["artist_tags_text"].str.split(" | ", regex=False)
tags_long_artist = tags_long_artist.explode("raw_tag", ignore_index=True)

# Normalize tags
tags_long_artist["raw_tag"] = tags_long_artist["raw_tag"].astype(str).str.strip().str.lower()

display(tags_long_artist.head())

,artist_id,artist_tags_text,raw_tag
0,15,european | film composer | french | soundtrack,european
1,15,european | film composer | french | soundtrack,film composer
2,15,european | film composer | french | soundtrack,french
3,15,european | film composer | french | soundtrack,soundtrack
4,25,indie rock | alternative rock | lo-fi | rock |...,indie rock


Here we apply the artist\-side mapping step, mirroring the album genre workflow\. Each normalized raw\_tag is mapped into a controlled canonical\_artist\_role cluster using our lookup dictionary, and unmapped/noise tags are dropped to keep only usable role signals\.

In [27]:
# ---- 4. Map raw tags to canonical artist role cluster ----
tags_long_artist["canonical_artist_role"] = tags_long_artist["raw_tag"].map(artist_tag_to_canon)
tags_mapped_artist = tags_long_artist.dropna(subset=["canonical_artist_role"]).copy()

display(tags_mapped_artist.head())

,artist_id,artist_tags_text,raw_tag,canonical_artist_role
1,15,european | film composer | french | soundtrack,film composer,media_composer_orchestral
3,15,european | film composer | french | soundtrack,soundtrack,media_composer_orchestral
4,25,indie rock | alternative rock | lo-fi | rock |...,indie rock,rock_artist_band
5,25,indie rock | alternative rock | lo-fi | rock |...,alternative rock,rock_artist_band
7,25,indie rock | alternative rock | lo-fi | rock |...,rock,rock_artist_band


Now that artist tags are normalized and mapped into a controlled set of role clusters, we materialize the result at the artist grain\. This pivot converts the long tag table into a wide “artist role append” table—one row per artist\_id, with boolean flags indicating which canonical role clusters are present\.

In [28]:
# ---- 5. Build the artist_role_append table at artist grain (boolean flag per canonical cluster) ----
artist_role_append = tags_mapped_artist.assign(present=1).pivot_table(
    index="artist_id",
    columns="canonical_artist_role",
    values="present",
    aggfunc="max",
    fill_value=0
).reset_index()

# Ensure all expected columns exist, even if a cluster didn't appear in the data
for c in ARTIST_ROLE_COLS:
    if c not in artist_role_append.columns:
        artist_role_append[c] = 0
    artist_role_append[c] = artist_role_append[c].astype(bool)

display(artist_role_append.head())

canonical_artist_role,artist_id,electronic_producer,jazz_traditional,media_composer_orchestral,pop_vocalist,rock_artist_band,world_folk_regional
0,1,True,True,True,True,True,True
1,15,False,False,True,False,False,False
2,25,False,False,False,False,True,False
3,35,False,True,False,True,True,True
4,70,False,False,True,False,False,False


Findings: The resulting flags behave as intended: artists can map to multiple role clusters \(e\.g\., artist\_id 1\), while many map cleanly to a single dominant cluster \(e\.g\., media\_composer\_orchestral for artist\_id 15 and 70\)\.

We merge the canonical artist\-role flags back onto the master artists\_df, preserving all artists via a left join and enforcing a many\-to\-one relationship \(m:1\) to guard against accidental duplication\. Artists without mapped tags are explicitly set to False across all role clusters, ensuring clean boolean fields for downstream modeling\.

In [29]:
# ---- 6. Merge back to artist_df (optional) ----
artists_df_w_roles = artists_df.merge(
    artist_role_append,
    on="artist_id",
    how="left",
    validate="m:1"
)

# Fill missing role flags with False (artists with no tags or no mapped tags)
for c in ARTIST_ROLE_COLS:
    artists_df_w_roles[c] = artists_df_w_roles[c].fillna(False).astype(bool)

display(artists_df_w_roles.head())

,artist_id,artist_mbid,name,sort_name,artist_type,gender,artist_comment,begin_date_year,begin_date_month,begin_date_day,...,isni_text,ipi_text,film_ct_in_spine,soundtrack_release_group_ct_in_spine,electronic_producer,jazz_traditional,media_composer_orchestral,pop_vocalist,rock_artist_band,world_folk_regional
0,15,974fa09f-8761-4fbb-b156-390417d125ae,Éric Serra,"Serra, Éric",Person,Male,French bassist and score composer,1959.0,9.0,9.0,...,0000000121028246,00044384581,4,3,False,False,True,False,False,False
1,25,36bfa85f-737b-41db-a8fc-b8825850ffc3,Pavement,Pavement,Group,NaN,US indie rock band,1989.0,NaN,NaN,...,0000000106573203,NaN,1,1,False,False,False,False,True,False
2,35,05755bf1-380c-487f-983f-d1a02401fa28,Cat Power,Cat Power,Person,Female,NaN,1972.0,1.0,21.0,...,0000000114707988 | 0000000368591153,NaN,1,1,False,True,False,True,True,True
3,70,3bd08e76-bf53-4b16-82ee-63f95ca5cdae,Harold Faltermeyer,"Faltermeyer, Harold",Person,Male,NaN,1952.0,10.0,5.0,...,000000011475360X,00040121057 | 00076212489 | 00219670949,1,1,False,False,True,False,False,False
4,94,53b106e7-0cc6-42cc-ac95-ed8d30a3a98e,John Williams,"Williams, John",Person,Male,American score composer,1932.0,2.0,8.0,...,0000000123479198,00032981579,7,7,False,True,True,False,False,False


Findings: The merged results look consistent with expectations: well\-known score composers \(e\.g\., Éric Serra, Harold Faltermeyer, John Williams\) are correctly flagged as media\_composer\_orchestral, while bands and singer\-songwriters map to rock, pop, or jazz clusters as appropriate\. Artists can legitimately span multiple clusters, reflecting real stylistic breadth rather than rule leakage\.

### IV\.3 Append to wide table

Next we propagate the artist\-role clusters we built at the artist grain onto the track\-grained wide table—but in a way that doesn’t blow up row counts\. The key move is to define a single “primary” album artist once per track row \(by taking the first artist in the pipe\-delimited artist list\), QA that it’s consistent within each album, and then use that stable key for clean, repeatable album\-level enrichment\.

In [30]:
# --------------------------------------------------------------------------
# BIG PICTURE: Attach Artist-role flags (artist-grained) to the track-grained
# wide table WITHOUT exploding rows.
#
# Strategy mirrors the Last.fm bridge:
#   0) Precompute primary artist fields on wide_mrg_df (e.g., primary_artist_id),
#      so we don't repeat string parsing in every enrichment step.
#   1) Collapse to one row per album (release_group_mbid), keeping primary_artist_id.
#   2) Join primary_artist_id to artist_role_append (one row per artist).
#   3) Prefix role flags with 'Artist_' so they stay distinct from album_ genres.
#   4) Merge album-level artist role flags back to wide_mrg_df on release_group_mbid.
# --------------------------------------------------------------------------

# STEP 0: Pull a single “primary” album artist onto every track row
# Why:
# - album_artist_* fields are stored as pipe-delimited lists ("id1 | id2 | ...")
# - we keep the *first* artist as the default “album artist” for album-level enrichments
# - doing this once up front keeps later joins clean and avoids repeating string parsing

wide_mrg_df = wide_mrg_df.copy()  # don’t mutate the original wide table in-place

# --- primary_artist_id ---
# Split on pipes with optional whitespace, then take the first token (index 0).
# Use to_numeric(errors="coerce") so blanks / weird tokens become <NA> instead of crashing.
primary_artist_id_raw = (
    wide_mrg_df["album_artist_ids_text"]
      .astype("string")
      .str.strip()
      .str.split(r"\s*\|\s*", regex=True)
      .str[0]
)

# Nullable integer dtype ("Int64") plays nicely with missing values.
wide_mrg_df["primary_artist_id"] = pd.to_numeric(
    primary_artist_id_raw, errors="coerce"
).astype("Int64")

# --- primary_artist (name) ---
# Same logic for the human-readable name list (handy for spot checks + QA).
wide_mrg_df["primary_artist"] = (
    wide_mrg_df["album_artist_names_text"]
      .astype("string")
      .str.strip()
      .str.split(r"\s*\|\s*", regex=True)
      .str[0]
)

# --- Quick QA ---
# 1) Coverage: do we actually have primary artist values most of the time?
print("primary_artist_id missing:", wide_mrg_df["primary_artist_id"].isna().mean())
print("primary_artist missing:", wide_mrg_df["primary_artist"].isna().mean())

# 2) Consistency: within a release_group (album), primary_artist_id should not vary across tracks.
# If this is >0, it usually means the wide table has inconsistent artist lists across track rows.
inconsistent = (
    wide_mrg_df.groupby("release_group_mbid")["primary_artist_id"]
    .nunique(dropna=True)
)
print("albums with >1 primary_artist_id:", (inconsistent > 1).sum())


primary_artist_id missing: 0.0
primary_artist missing: 0.0
albums with >1 primary_artist_id: 0


Primary artist coverage is complete: primary\_artist\_id and primary\_artist are populated for 100% of rows\. Just as importantly, the primary artist assignment is internally consistent—no albums show multiple primary\_artist\_ids across their track rows—so the album\-level join will be stable and non\-duplicative\.

Now that each track row has a stable primary\_artist\_id, we can attach artist\-role signals without inflating the dataset\. We first collapse to one row per album \(release\_group\_mbid → primary\_artist\_id\), join in the artist\-role flags at the artist grain, prefix those columns to keep them distinct from album genres, and then merge the album\-level role flags back onto the full track\-grained wide table\.

In [31]:
ARTIST_ROLE_COLS = list(CANONICAL_ARTIST_ROLE_MAP.keys())

# 1) One row per album: release_group_mbid → primary_artist_id
#    (primary_artist_id is already computed in Step 0)
album_primary_artist = (
    wide_mrg_df[["release_group_mbid", "primary_artist_id"]]
    .drop_duplicates(subset=["release_group_mbid"])
)


# 2) Prefix artist role flags (keep artist_id as the join key for now)
artist_role_append_renamed = artist_role_append.rename(
    columns={c: f"Artist_{c}" for c in ARTIST_ROLE_COLS}
)

# 3) Join role flags onto primary_artist_id (artist-grain join)
album_primary_artist_roles = album_primary_artist.merge(
    artist_role_append_renamed.rename(columns={"artist_id": "primary_artist_id"}),
    on="primary_artist_id",
    how="left",
    validate="m:1"
)

# 4) Keep only the columns we want to append back to wide (album grain)
cols_to_add = ["release_group_mbid", "primary_artist_id"] + [f"Artist_{c}" for c in ARTIST_ROLE_COLS]
album_primary_artist_roles = album_primary_artist_roles[cols_to_add].rename(
    columns={"primary_artist_id": "album_primary_artist_id"}
)

# Optional: fill missing role flags as False (albums whose primary artist had no tags / no mappings)
for c in [f"Artist_{x}" for x in ARTIST_ROLE_COLS]:
    if c in album_primary_artist_roles.columns:
        album_primary_artist_roles[c] = album_primary_artist_roles[c].fillna(False).astype(bool)

# 5) Merge back into the track-grained wide table
before = wide_mrg_df.shape

wide_mrg_df2 = wide_mrg_df.merge(
    album_primary_artist_roles,
    on="release_group_mbid",
    how="left",
    validate="m:1"
)

print("wide_mrg_df shape before:", before)
print("wide_mrg_df shape after: ", wide_mrg_df2.shape)
display(wide_mrg_df2.columns)


wide_mrg_df shape before: (78992, 103)
wide_mrg_df shape after:  (78992, 110)


Index(['tmdb_id', 'film_title', 'film_adult', 'film_runtime_min',
       'film_genres', 'film_rating', 'film_vote_count', 'film_mpaa_rating',
       'film_original_title', 'film_language_name',
       ...
       'album_world_folk', 'primary_artist_id', 'primary_artist',
       'album_primary_artist_id', 'Artist_media_composer_orchestral',
       'Artist_pop_vocalist', 'Artist_rock_artist_band',
       'Artist_electronic_producer', 'Artist_jazz_traditional',
       'Artist_world_folk_regional'],
      dtype='object', length=110)

Findings: The artist\-role enrichment preserves row integrity: the wide table remains at 78,992 rows, confirming no accidental row expansion\. The merge simply appends seven new artist\-role flags \(plus album\_primary\_artist\_id\), cleanly extending the feature set from 103 to 110 columns without altering the track\-level grain\.

# V\. Write everything back out

In [32]:
out_path = "./pipeline/3.5.Albums_exploded_genre.csv"

album_merged_df.to_csv(out_path, index=False)

In [33]:
out_path = "./pipeline/3.5.Artists_exploded_genre.csv"

artists_df_w_roles.to_csv(out_path, index=False)

Tracks just carry over

In [34]:
import shutil

shutil.copy(
    "./pipeline/3.4.Tracks_canonical_identified_df.csv",
    "./pipeline/3.5.Tracks_exploded_genre.csv"
)



'./pipeline/3.5.Tracks_exploded_genre.csv'

In [35]:
out_path = "./pipeline/3.5.Wide_exploded_genre.csv"

wide_mrg_df2.to_csv(out_path, index=False)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b124131d-2024-4253-bb46-8043aed4b78f' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>